# Loading in the data

In [ ]:
import duckdb

con = duckdb.connect()

# Create a cleaned table directly from raw data
con.execute("""
    CREATE TABLE loans_raw AS
    SELECT *
    FROM read_csv_auto(
        '/content/accepted_2007_to_2018Q4.csv.gz',
        all_varchar=true,
        ignore_errors=true
    )
""")

# Preview
con.execute("SELECT * FROM loans_raw LIMIT 5").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,None,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,None,None,Cash,N,None,None,None,None,None,None
1,68355089,None,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,None,None,Cash,N,None,None,None,None,None,None
2,68341763,None,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,None,None,Cash,N,None,None,None,None,None,None
3,66310712,None,35000.0,35000.0,35000.0,60 months,14.85,829.9,C,C5,...,None,None,Cash,N,None,None,None,None,None,None
4,68476807,None,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,None,None,Cash,N,None,None,None,None,None,None


Clean the Data

In [ ]:
cols = con.execute("SELECT * FROM loans_raw LIMIT 1").df().columns
print(list(cols))

['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq',

In [ ]:
con.execute("""
    CREATE TABLE loans_clean AS
    SELECT
        id,
        loan_amnt,
        term,
        int_rate,
        installment,
        loan_status,

        emp_length,
        home_ownership,
        annual_inc,
        verification_status,

        purpose,
        grade,
        sub_grade,
        application_type,

        dti,
        fico_range_low,
        fico_range_high,
        open_acc,
        pub_rec,
        revol_bal,
        revol_util,
        total_acc,
        delinq_2yrs,
        inq_last_6mths

    FROM loans_raw
    WHERE loan_status IS NOT NULL
""")

Create Target

In [ ]:
con.execute("""
    CREATE TABLE loans_final AS
    SELECT *,
        CASE
            WHEN loan_status = 'Charged Off' THEN 1
            ELSE 0
        END AS default_flag
    FROM loans_clean
""")

In [ ]:
con.execute("""
    COPY loans_final
    TO '/content/loans_dataset.csv.gz'
    (HEADER, DELIMITER ',', COMPRESSION 'gzip')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Create tables


In [ ]:
con.execute("""
    CREATE TABLE loans AS
    SELECT
        id,
        loan_amnt,
        term,
        int_rate,
        installment,
        loan_status,
        default_flag
    FROM loans_final
""")

In [ ]:
con.execute("""
    CREATE TABLE borrowers AS
    SELECT
        id,
        emp_length,
        home_ownership,
        annual_inc
    FROM loans_final
""")

In [ ]:
con.execute("""
    CREATE TABLE credit AS
    SELECT
        id,
        fico_range_low,
        fico_range_high,
        open_acc,
        pub_rec,
        revol_bal,
        revol_util,
        total_acc,
        dti
    FROM loans_final
""")

In [ ]:
con.execute("""
    CREATE TABLE loan_details AS
    SELECT
        id,
        purpose,
        grade,
        sub_grade
    FROM loans_final
""")

In [ ]:
# loans
con.execute("""
    COPY loans TO '/content/loans.csv' (HEADER, DELIMITER ',')
""")

# borrowers
con.execute("""
    COPY borrowers TO '/content/borrowers.csv' (HEADER, DELIMITER ',')
""")

# credit
con.execute("""
    COPY credit TO '/content/credit.csv' (HEADER, DELIMITER ',')
""")

# loan_details
con.execute("""
    COPY loan_details TO '/content/loan_details.csv' (HEADER, DELIMITER ',')
""")


In [ ]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN loan_amnt IS NULL THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS loan_amnt_missing,
    SUM(CASE WHEN int_rate IS NULL THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS int_rate_missing,
    SUM(CASE WHEN installment IS NULL THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS installment_missing,
    SUM(CASE WHEN annual_inc IS NULL THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS annual_inc_missing,
    SUM(CASE WHEN fico_range_low IS NULL THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS fico_low_missing,
    SUM(CASE WHEN fico_range_high IS NULL THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS fico_high_missing,
    SUM(CASE WHEN open_acc IS NULL THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS open_acc_missing,
    SUM(CASE WHEN revol_util IS NULL THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS revol_util_missing,
    SUM(CASE WHEN total_acc IS NULL THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS total_acc_missing,
    SUM(CASE WHEN dti IS NULL THEN 1 ELSE 0 END)*1.0 / COUNT(*) AS dti_missing
FROM loans_final;
""").fetchdf()

,total_rows,loan_amnt_missing,int_rate_missing,installment_missing,annual_inc_missing,fico_low_missing,fico_high_missing,open_acc_missing,revol_util_missing,total_acc_missing,dti_missing
0,116082,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000388,0.0,0.000017


In [ ]:
con.execute("""
SELECT
    AVG(CAST(loan_amnt AS DOUBLE)) AS avg_loan,
    STDDEV(CAST(loan_amnt AS DOUBLE)) AS std_loan,

    AVG(CAST(annual_inc AS DOUBLE)) AS avg_income,
    STDDEV(CAST(annual_inc AS DOUBLE)) AS std_income,

    AVG(CAST(dti AS DOUBLE)) AS avg_dti,
    STDDEV(CAST(dti AS DOUBLE)) AS std_dti
FROM loans_final;
""").fetchdf()


,avg_loan,std_loan,avg_income,std_income,avg_dti,std_dti
0,15098.540902,8643.756651,78610.470445,87805.721798,19.263199,9.507559
